# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```

**What this notebook does**
1. Loads the environment and checks the keys
2. Creates a persistent ChromaDB database (`chromadb/`) and the collection `udaplay`
3. Wraps everything in a reusable `VectorStoreManager`
4. Loads and processes the 15 game JSON files and embeds them
5. Demonstrates semantic search

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"
print("Environment OK")

In [ ]:
import json
import chromadb
from chromadb.utils import embedding_functions

### VectorDB Instance

In [ ]:
# Keep this path: Part 2 loads the database from "chromadb" (inside project/starter)
chroma_client = chromadb.PersistentClient(path="chromadb")
print("Chroma client ready. Existing collections:", [c.name for c in chroma_client.list_collections()])

### Collection

In [ ]:
# Part 2 rebuilds this same embedding function when it loads the collection
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [ ]:
# Keep this name: Part 2 loads the collection "udaplay"
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)
print("Collection:", collection.name, "| documents so far:", collection.count())

### Reusable Vector Store Manager

A small class that owns the client, embedding function and collection, so loading data and searching
can be reused anywhere (including from the agent in Part 2).

In [ ]:
class VectorStoreManager:
    """Reusable wrapper around a persistent ChromaDB collection."""

    def __init__(self, path="chromadb", collection_name="udaplay"):
        self.client = chromadb.PersistentClient(path=path)
        self.embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
            api_key_env_var="OPENAI_API_KEY",
            api_base=os.getenv("OPENAI_BASE_URL"),
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_fn,
        )

    @staticmethod
    def game_to_text(game):
        """Text that gets embedded. Name/platform/year/genre/publisher are included
        so questions about any of them can match semantically."""
        return (
            f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
            f"Genre: {game['Genre']}. Publisher: {game['Publisher']}. {game['Description']}"
        )

    def load_games(self, data_dir="games"):
        """Load every JSON file in data_dir and upsert it (safe to re-run)."""
        ids, docs, metas = [], [], []
        for file_name in sorted(os.listdir(data_dir)):
            if not file_name.endswith(".json"):
                continue
            with open(os.path.join(data_dir, file_name), "r", encoding="utf-8") as f:
                game = json.load(f)
            ids.append(os.path.splitext(file_name)[0])   # file name (like 001) as ID
            docs.append(self.game_to_text(game))
            metas.append(game)
        # upsert (not add) so re-running the notebook does not create duplicate-ID errors
        self.collection.upsert(ids=ids, documents=docs, metadatas=metas)
        return len(ids)

    def search(self, query, n_results=3):
        """Semantic search. Returns a list of dicts with the game fields and the distance."""
        res = self.collection.query(query_texts=[query], n_results=n_results)
        out = []
        for i in range(len(res["ids"][0])):
            item = dict(res["metadatas"][0][i])
            item["id"] = res["ids"][0][i]
            item["distance"] = res["distances"][0][i]
            out.append(item)
        return out

    def count(self):
        return self.collection.count()

### Add documents
Load and process the game JSON files, embed them and store them in the persistent collection.

In [ ]:
# Run this notebook from project/starter: the game files are in ./games
store = VectorStoreManager(path="chromadb", collection_name="udaplay")
n = store.load_games("games")
print(f"Processed {n} game files -> collection now holds {store.count()} documents")

Quick look at the processed documents (what was actually embedded):

In [ ]:
peek = store.collection.get(limit=15)
for _id, doc in zip(peek["ids"], peek["documents"]):
    print(_id, "|", doc[:140])

### Demonstrate semantic search
The queries below don't need to share exact words with the documents; the embeddings do the matching.

In [ ]:
test_queries = [
    "Who published Grand Theft Auto: San Andreas?",
    "When was Pokemon Gold and Silver released?",
    "What platform was Super Mario 64 launched on?",
    "a realistic racing simulator",
    "first 3D platformer from Nintendo",
]

for q in test_queries:
    print("=" * 90)
    print("QUERY:", q)
    for r in store.search(q, n_results=3):
        print(f"  dist={r['distance']:.3f} | {r['Name']} | {r['Platform']} | {r['YearOfRelease']} | {r['Publisher']}")

### Notes
- The database lives in `chromadb/` and the collection is called `udaplay`. **Part 2 loads exactly these.** Don't delete the folder; if you do, re-run this notebook.
- Telemetry messages from ChromaDB (`Failed to send telemetry event ...`) are harmless.